In [0]:
events_df = spark.read.table("workspace.ecommerce.events_delta")
events_df.printSchema()

In [0]:
from pyspark.sql.functions import when, col

ratings_df = events_df.withColumn(
    "rating",
    when(col("event_type") == "purchase", 3.0)
    .when(col("event_type") == "cart", 2.0)
    .otherwise(1.0)
)

In [0]:
als_data = ratings_df.select(
    col("user_id").alias("userId"),
    col("product_id").alias("itemId"),
    col("rating")
).dropna()

In [0]:
from pyspark.sql.functions import sum as _sum

als_data = als_data.groupBy("userId", "itemId").agg(
    _sum("rating").alias("rating")
)

In [0]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="itemId",
    ratingCol="rating",
    rank=10,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
    nonnegative=True
)

als_model = als.fit(als_data)

In [0]:
user_recs = als_model.recommendForAllUsers(5)

In [0]:
from pyspark.sql.functions import explode

recommendations = user_recs.select(
    "userId",
    explode("recommendations").alias("rec")
).select(
    "userId",
    col("rec.itemId").alias("product_id"),
    col("rec.rating").alias("score")
)